# Chapter 2 — Spectral Clustering & Normalized Cut

This chapter covers two spectral graph partitioning algorithms from the thesis:
- **Spectral Clustering** — embeds nodes via Laplacian eigenvectors, then applies k-means
- **Normalized Cut** — minimises a balanced cut criterion via the normalised Laplacian

---

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from scipy.linalg import eigh
from numpy import sqrt

## 2.1 Spectral Clustering

### K-Means Algorithm

K-means partitions a set of points $P = \{p_i\}_{i=1}^n \subset \mathbb{R}^k$ into $k$ clusters by minimising the total intra-cluster squared Euclidean distance.
The algorithm alternates between two steps until convergence:

1. **Assign** each point to the nearest centroid: $i_0 = \arg\min_{i=1,\ldots,k}\|p - c_i\|^2$
2. **Update** each centroid as the mean of its assigned points: $c_i' = \frac{1}{|P_i|}\sum_{p \in P_i} p$

The pseudocode and implementation follow Algorithm 1 of the thesis.

In [ ]:
# ---- K-Means function ----
def kmeans(points, k, max_iters=100):
    indices = np.random.choice(len(points), k, replace=False)
    centroids = points[indices]
    initial_centroids = centroids.copy()
    for it in range(max_iters):
        distances = np.linalg.norm(points[:, np.newaxis, :] - centroids[np.newaxis, :, :], axis=2)
        cluster_assignments = np.argmin(distances, axis=1)
        new_centroids = np.array([
            points[cluster_assignments == i].mean(axis=0) if np.any(cluster_assignments == i) else centroids[i]
            for i in range(k)
        ])
        if np.allclose(centroids, new_centroids):
            break
        centroids = new_centroids
    return centroids, cluster_assignments, it+1, initial_centroids

# ---- Plot clusters ----
def plot_clusters(points, centroids, cluster_assignments, k, title):
    colors = ['red', 'green', 'blue', 'yellow', 'purple']
    for i in range(k):
        cluster_points = points[cluster_assignments == i]
        plt.scatter(cluster_points[:, 0], cluster_points[:, 1], c=colors[i % len(colors)], label=f'Cluster {i+1}')
    plt.scatter(centroids[:, 0], centroids[:, 1], c='black', marker='x', s=100, label='Centroids')
    plt.title(title)
    plt.legend()
    plt.show()

### Example 2.1 — 20 points, k=2

Clustering of 20 random points into 2 groups. The crosses mark the initial centroids.

In [ ]:
# ---- 20 random points, k=2 ----
points_small = np.random.rand(20, 2)
k_small = 2

centroids_s, assignments_s, iters_s, init_s = kmeans(points_small, k_small)

# ---- Before clustering ----
plt.scatter(points_small[:, 0], points_small[:, 1], c='grey', alpha=0.6)
plt.scatter(init_s[:, 0], init_s[:, 1], c='black', marker='x', s=100, label='Initial Centroids')
plt.title('Before clustering, initial centroids')
plt.legend()
plt.show()

# ---- After clustering ----
plot_clusters(points_small, centroids_s, assignments_s, k_small, 'After clustering')

### Example 2.2 — 1000 points, k=5

Clustering of 1000 random points into 5 groups.

In [ ]:
# ---- 1000 random points, k=5 ----
points = np.random.rand(1000, 2)
k = 5

centroids, cluster_assignments, iterations, initial_centroids = kmeans(points, k)

# ---- Before clustering ----
plt.scatter(points[:, 0], points[:, 1], c='grey', alpha=0.6)
plt.scatter(initial_centroids[:, 0], initial_centroids[:, 1], c='black', marker='x', s=100, label='Initial Centroids')
plt.title('Before clustering, initial centroids')
plt.legend()
plt.show()

print('Final centroids:\n', centroids)
print('Iterations:', iterations)

# ---- After clustering ----
plot_clusters(points, centroids, cluster_assignments, k, 'After clustering')

### Spectral Clustering Algorithm

The key insight: the $k$ smallest eigenvectors of the Laplacian $L$ encode the cluster structure of the graph. Nodes are embedded in $\mathbb{R}^k$ using these eigenvectors and then clustered with k-means.

**Algorithm 2 (Spectral Clustering):**
1. Compute adjacency matrix $A$ and Laplacian $L = D - A$
2. Compute the $k$ smallest eigenvectors of $L$ — form matrix $U \in \mathbb{R}^{N \times k}$
3. Treat each row of $U$ as a point in $\mathbb{R}^k$
4. Apply k-means on these $N$ points

The implementation follows the Python code in Algorithm 2 of the thesis.

In [ ]:
# ---- Step 1: Random graph ----
N = 8
M = 14
G = nx.gnm_random_graph(n=N, m=M, seed=42)
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, cmap=plt.cm.Set1, edge_color='grey')
plt.show()

# ---- Step 2: Adjacency matrix and Laplacian ----
A = nx.to_numpy_array(G, dtype=int)
degrees = np.sum(A, axis=1)
D = np.diag(degrees)
L = D - A

# ---- Step 3: Eigendecomposition ----
eigenvalues, eigenvectors = np.linalg.eigh(L)
k = 2
top_k_eigenvectors = eigenvectors[:, :k]
top_k_eigenvalues = eigenvalues[:k]

# ---- Step 4: KMeans on rows of N x k matrix ----
X = top_k_eigenvectors
kmeans_model = KMeans(n_clusters=k, n_init=1500, random_state=0).fit(X)
labels = kmeans_model.labels_

# ---- Step 5: Output ----
print('Cluster assignments for each node:', labels)
print('Laplacian matrix L:')
print(L)
print('Top k eigenvalues:', top_k_eigenvalues)
print('Top k eigenvectors:', top_k_eigenvectors)

# ---- Step 6: Draw graphs ----
nx.draw(G, with_labels=True, node_color=labels, cmap=plt.cm.Set1, edge_color='grey')
plt.show()

pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_color=labels, cmap=plt.cm.Set1, edge_color='grey')
plt.show()

### Example 2.4 — N=12, M=21, k=2 and k=4

When $k=2$ the partition can be unbalanced — one cluster may contain a single node. Increasing $k$ produces a more balanced result.

In [ ]:
# ---- Graph N=12, M=21 ----
N2 = 12
M2 = 21
G2 = nx.gnm_random_graph(n=N2, m=M2, seed=42)
pos2 = nx.spring_layout(G2, seed=42)

A2_mat = nx.to_numpy_array(G2, dtype=int)
degrees2 = np.sum(A2_mat, axis=1)
D2 = np.diag(degrees2)
L2 = D2 - A2_mat

eigenvalues2, eigenvectors2 = np.linalg.eigh(L2)

nx.draw(G2, pos2, with_labels=True, node_color='steelblue', edge_color='grey')
plt.title('Original Graph (N=12, M=21)')
plt.show()

# ---- k=2 ----
k2 = 2
labels_k2 = KMeans(n_clusters=k2, n_init=1500, random_state=0).fit_predict(eigenvectors2[:, :k2])
nx.draw(G2, pos2, with_labels=True, node_color=labels_k2, cmap=plt.cm.Set1, edge_color='grey')
plt.title('Spectral Clustering k=2')
plt.show()

# ---- k=4 ----
k4 = 4
labels_k4 = KMeans(n_clusters=k4, n_init=1500, random_state=0).fit_predict(eigenvectors2[:, :k4])
nx.draw(G2, pos2, with_labels=True, node_color=labels_k4, cmap=plt.cm.Set1, edge_color='grey')
plt.title('Spectral Clustering k=4')
plt.show()

## 2.2 Normalized Cut

Spectral Clustering minimises the raw cut size, which tends to isolate weakly connected nodes. **Normalized Cut** (Shi & Malik, 2000) fixes this by normalising the cut relative to total connectivity, penalising unbalanced splits.

Given a partition into sets $A$ and $B$:
$$\operatorname{cut}(A, B) = \sum_{u \in A,\, v \in B} w(u,v)$$
$$\operatorname{assoc}(A, V) = \sum_{u \in A,\, t \in V} w(u,t)$$
$$\operatorname{Ncut}(A, B) = \frac{\operatorname{cut}(A,B)}{\operatorname{assoc}(A,V)} + \frac{\operatorname{cut}(B,A)}{\operatorname{assoc}(B,V)}$$

Minimising Ncut is NP-hard in the discrete case. The continuous relaxation leads to the generalised eigenvalue problem $(D - W)\mathbf{y} = \lambda D\mathbf{y}$, equivalent to finding eigenvectors of the normalised Laplacian $L_N = D^{-1/2}(D-W)D^{-1/2}$.

The partition is recovered from the **Fiedler vector** (2nd smallest eigenvector):
$$\mathbf{y}_1 = D^{-1/2}\mathbf{z}_1, \quad A = \{i : y_1(i) \geq 0\},\quad B = \{i : y_1(i) < 0\}$$

The implementation follows Algorithm 3 of the thesis.

In [ ]:
# ---- Normalized Cut function ----
def normalized_cut_with_metrics(G):
    W = nx.to_numpy_array(G, weight='weight')
    degrees = W.sum(axis=1)
    D = np.diag(degrees)
    D_inv_sqrt = np.diag(1.0 / sqrt(degrees))
    L = D - W
    L_sym = np.eye(len(W)) - D_inv_sqrt @ W @ D_inv_sqrt
    eigvals, eigvecs = eigh(L_sym)
    z1 = eigvecs[:, 1]
    y1 = D_inv_sqrt @ z1
    partition = y1 >= 0
    set_A = np.where(partition)[0]
    set_B = np.where(~partition)[0]

    assoc_A_V = np.sum(W[set_A, :])
    assoc_B_V = np.sum(W[set_B, :])
    cut_AB = np.sum(W[np.ix_(set_A, set_B)])
    ncut = (cut_AB / assoc_A_V) + (cut_AB / assoc_B_V)

    return set_A, set_B, z1, y1, assoc_A_V, assoc_B_V, cut_AB, ncut

### Example 2.5 — N=8, M=14

In [ ]:
# ---- Graph N=8, M=14 ----
N_nc1 = 8
M_nc1 = 14
G_nc1 = nx.gnm_random_graph(N_nc1, M_nc1, seed=42)

A_nc1, B_nc1, z1_nc1, y1_nc1, assoc_A1, assoc_B1, cut1, ncut1 = normalized_cut_with_metrics(G_nc1)

pos_nc1 = nx.spring_layout(G_nc1, seed=42)

# ---- Original graph ----
nx.draw(G_nc1, pos_nc1, with_labels=True, node_color='lightblue', edge_color='gray')
plt.title('Original Graph')
plt.show()

# ---- After Normalized Cut ----
colors1 = ['orange' if node in A_nc1 else 'green' for node in G_nc1.nodes()]
nx.draw(G_nc1, pos_nc1, with_labels=True, node_color=colors1, edge_color='gray')
plt.title('After Normalized Cut')
plt.show()

print('Nodes in A:', list(map(int, A_nc1)))
print('Nodes in B:', list(map(int, B_nc1)))
print('assoc(A, V):', assoc_A1)
print('assoc(B, V):', assoc_B1)
print('cut(A, B):', cut1)
print('Ncut:', round(ncut1, 4))

### Example 2.6 — N=21, M=35

In [ ]:
# ---- Graph N=21, M=35 ----
N_nc2 = 21
M_nc2 = 35
G_nc2 = nx.gnm_random_graph(N_nc2, M_nc2, seed=30)

A_nc2, B_nc2, z1_nc2, y1_nc2, assoc_A2, assoc_B2, cut2, ncut2 = normalized_cut_with_metrics(G_nc2)

pos_nc2 = nx.spring_layout(G_nc2, seed=42)

# ---- Original graph ----
nx.draw(G_nc2, pos_nc2, with_labels=True, node_color='lightblue', edge_color='gray')
plt.title('Original Graph')
plt.show()

# ---- After Normalized Cut ----
colors2 = ['orange' if node in A_nc2 else 'green' for node in G_nc2.nodes()]
nx.draw(G_nc2, pos_nc2, with_labels=True, node_color=colors2, edge_color='gray')
plt.title('After Normalized Cut')
plt.show()

print('Nodes in A:', list(map(int, A_nc2)))
print('Nodes in B:', list(map(int, B_nc2)))
print('assoc(A, V):', assoc_A2)
print('assoc(B, V):', assoc_B2)
print('cut(A, B):', cut2)
print('Ncut:', round(ncut2, 4))

---
## Summary

| Method | Core idea | Known limitation |
|--------|-----------|------------------|
| Spectral Clustering | Embed via Laplacian eigenvectors + k-means | Can produce unbalanced partitions |
| Normalized Cut | Minimise Ncut via Fiedler vector of $L_N$ | Penalises isolation of small sets |

**Next:** [Chapter 3 — Kernighan-Lin](03_kernighan_lin.ipynb)